In [1]:
import torch
import cv2
import os
import matplotlib 
from matplotlib import pyplot as plt 
import imghdr
import numpy as np 
import torch.nn as nn
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.optim as optim
torch.backends.cudnn.benchmark = True

In [2]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    #transforms.RandomResizedCrop(224),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomRotation(10),
    # transforms.ColorJitter(    
    #     brightness=0.2,
    #     contrast=0.2,
    #     saturation=0.2
    # ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = mean,
        std = std
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),          # Resize
    transforms.CenterCrop(224),             
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,
                         std=std)
])

In [3]:
train_dataset = ImageFolder(
    root = 'tomato_data/train',
    transform = train_transforms
)

val_dataset = ImageFolder(
    root = 'tomato_data/val',
    transform = val_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True, 
    num_workers = 4
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 64,
    shuffle = False, 
    num_workers = 4
)

In [4]:
print(train_dataset.classes)
print(f"Training images: {len(train_dataset)}")
print(val_dataset.classes)
print(f"Validation images: {len(val_dataset)}")

['Damaged', 'Old', 'Ripe', 'Unripe']
Training images: 6500
['Damaged', 'Old', 'Ripe', 'Unripe']
Validation images: 724


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.get_device_name(0))

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 4) 

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [6]:
scaler = torch.cuda.amp.GradScaler()

num_epochs = 5

for epoch in range(num_epochs):


    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = train_correct / train_total



    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total


    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")



Epoch [1/5]
Train Loss: 0.2988 | Train Acc: 0.9145
Val   Loss: 0.0963 | Val   Acc: 0.9641

Epoch [2/5]
Train Loss: 0.0363 | Train Acc: 0.9889
Val   Loss: 0.0495 | Val   Acc: 0.9890

Epoch [3/5]
Train Loss: 0.0270 | Train Acc: 0.9912
Val   Loss: 0.0499 | Val   Acc: 0.9820

Epoch [4/5]
Train Loss: 0.0110 | Train Acc: 0.9972
Val   Loss: 0.0537 | Val   Acc: 0.9848

Epoch [5/5]
Train Loss: 0.0063 | Train Acc: 0.9982
Val   Loss: 0.0589 | Val   Acc: 0.9876


In [8]:
torch.save(model.state_dict(), "best_resnet50_tomato.pth")
print("model saved")

model saved
